In [1]:
from pathlib import Path
def load_text(path):
    return Path(path).read_text(encoding='utf-8').strip()

textA =load_text("gemini.txt")
textB =load_text("mistral.txt")

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(lowercase=True, stop_words='english', ngram_range=(1,2))

X= tfidf.fit_transform([textA, textB])
tfidf_cosine= cosine_similarity(X[0], X[1])[0][0]

In [7]:
from rapidfuzz.fuzz import ratio

char_similarity = ratio(textA, textB) / 100.0

In [ ]:
from sentence_transformers import SentenceTransformer, util 
embedded = SentenceTransformer('all-MiniLM-L6-v2')

eA = embedded.encode(textA, convert_to_tensor=True, normalize_embeddings=True)
eB = embedded.encode(textB, convert_to_tensor=True, normalize_embeddings=True)

semantic_cosine = util.cos_sim(eA, eB).item()

In [17]:
import re
import textstat

def text_stats(t):
    sentences = [s for s in re.split(r'[.!?]+', t) if s.strip()]
    words = re.findall(r'\b\w+\b', t.lower())

    return{
        "char": len(t),
        "words": len(words),
        "sentences": len(sentences),
        "avg_words_per_sentence": round(len(words)/max(1, len(sentences), 2)),
        "flesch_reading_ease": (textstat.flesch_reading_ease(t),2),
        "fk_grade": (textstat.flesch_kincaid_grade(t),2),
    }

statsA = text_stats(textA)
statsB = text_stats(textB)

In [23]:
import pandas as pd
comparison = pd.DataFrame(
    {
        "Gemini": statsA,
        "Mistral": statsB,
    }
)

similarities = pd.DataFrame({
    "Metric": ["TF-IDF Cosine Similarity", "Character-level Similarity", "Semantic Cosine Similarity"],
    "Value": [tfidf_cosine, char_similarity, semantic_cosine]
})

print("Text Statistics Comparison:")
display(comparison)
print("\nSimilarity Metrics:")
display(similarities)

Text Statistics Comparison:


,Gemini,Mistral
char,3834,4952
words,486,599
sentences,22,58
avg_words_per_sentence,22,10
flesch_reading_ease,"(1.9165819209039796, 2)","(16.692624331248595, 2)"
fk_grade,"(18.575714285714287, 2)","(14.04719049054329, 2)"



Similarity Metrics:


,Metric,Value
0,TF-IDF Cosine Similarity,0.497203
1,Character-level Similarity,0.449579
2,Semantic Cosine Similarity,0.932629
